# Lesson 15b: Efficient and Scalable Deep Learning — Practical

15a derived why mixed precision, quantisation and distillation work: the
arithmetic behind memory, underflow, scale/zero-point, and the distillation
loss. This notebook runs each one as a real experiment with PyTorch's
production implementations — `torch.autocast`, `torch.quantization`, and a
real teacher/student training loop — on the same tiny CIFAR-10 network, and
reports what actually happens on this machine rather than what the
techniques promise in general. That distinction matters: mixed precision's
speed benefit, in particular, depends on hardware this notebook's CPU does
not have, and the honest result below says so directly.

By the end of this notebook you will have:
- trained with `torch.autocast`'s automatic mixed precision and measured
  both the memory difference (exactly 2x, unconditionally) and the speed
  difference (hardware-dependent — measured directly, not assumed),
- applied PyTorch's dynamic quantisation to a trained model and reported the
  resulting size, inference latency, and accuracy change,
- distilled a small CNN from a larger one with a real training loop and
  compared its accuracy against training the same small architecture
  directly, and
- collected every measurement above into one comparison table.

## Introduction

Three techniques, three different production APIs: `torch.autocast` wraps a
forward pass so PyTorch chooses a narrower dtype for matrix multiplications
automatically; `torch.quantization.quantize_dynamic` replaces a trained
model's `nn.Linear` layers with int8 versions after training, no
retraining required; distillation is not a library call at all, just a
different loss function inside an ordinary training loop, reproducing what
15a derived by hand. Each is measured here exactly as it would be measured
on a real project: before-and-after numbers on the same model and data, not
a description of what the technique is supposed to do.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, minibatch sampling,
# data subsampling) is reproducible.
import io
import os
import pathlib
import tempfile
import time
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### Loading a CIFAR-10 Subset

The same small subset as 15a — 2,000 training images, 500 held out — via
the Hugging Face parquet mirror.

In [ ]:
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])
    labels = df.iloc[idx]["label"].to_numpy()
    return images.transpose(0, 3, 1, 2), labels


X_train, y_train = load_cifar10_subset("train", 2000, seed=SEED)
X_test, y_test = load_cifar10_subset("test", 500, seed=SEED)
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)


def accuracy(net, X, y):
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()


print(f"train: {X_train_t.shape}, test: {X_test_t.shape}")

## Mixed Precision in Practice

`torch.autocast(device_type="cpu", dtype=torch.bfloat16)` wraps a forward
pass and lets PyTorch choose `bfloat16` for the operations that benefit from
it, exactly the mechanism 15a derived by hand. Two things are measured
separately, because they do not have to move together: the **memory** an
autocast tensor occupies (a property of the dtype alone — `bfloat16` is
always 2 bytes against `fp32`'s 4, unconditionally), and the **speed** of
computing with it (a property of the *hardware*: it is only faster where the
CPU or GPU has dedicated low-precision arithmetic units; on hardware without
them, casting to `bfloat16` and back adds overhead with nothing to offset
it).

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.relu(self.fc1(x.flatten(1)))
        return self.fc2(x)


amp_model = Net()
batch = X_train_t[:64]


def time_step(use_autocast, n=20):
    opt = torch.optim.Adam(amp_model.parameters(), lr=1e-3)
    for _ in range(3):  # warmup, excluded from timing
        opt.zero_grad()
        if use_autocast:
            with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
                loss = amp_model(batch).pow(2).mean()
        else:
            loss = amp_model(batch).pow(2).mean()
        loss.backward()
        opt.step()
    t0 = time.perf_counter()
    for _ in range(n):
        opt.zero_grad()
        if use_autocast:
            with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
                loss = amp_model(batch).pow(2).mean()
        else:
            loss = amp_model(batch).pow(2).mean()
        loss.backward()
        opt.step()
    return (time.perf_counter() - t0) / n


t_fp32 = time_step(use_autocast=False)
t_bf16 = time_step(use_autocast=True)
print(f"fp32:            {t_fp32*1000:.2f} ms/step")
print(f"bf16 (autocast): {t_bf16*1000:.2f} ms/step")
print(f"speed ratio (fp32 time / bf16 time): {t_fp32/t_bf16:.2f}x "
      f"({'faster' if t_bf16 < t_fp32 else 'SLOWER'} under autocast on this CPU)")

In [ ]:
# The memory difference, unlike the speed difference, does not depend on the hardware.
with torch.no_grad():
    out_fp32 = amp_model.conv1(batch)
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        out_bf16 = amp_model.conv1(batch)
bytes_fp32 = out_fp32.numel() * out_fp32.element_size()
bytes_bf16 = out_bf16.numel() * out_bf16.element_size()
print(f"conv1 activation: fp32 dtype={out_fp32.dtype} ({out_fp32.element_size()} B/elem, {bytes_fp32:,} B total)")
print(f"conv1 activation: autocast dtype={out_bf16.dtype} ({out_bf16.element_size()} B/elem, {bytes_bf16:,} B total)")
print(f"memory ratio: {bytes_fp32/bytes_bf16:.1f}x")
assert bytes_fp32 / bytes_bf16 == 2.0

The activation memory result is exactly what the dtype guarantees: 2 bytes
per element instead of 4, a clean 2x reduction, every time, on any hardware.
The *speed* result on this machine is the opposite of the usual pitch for
mixed precision — measured, not assumed, `bfloat16` autocast ran slower than
plain `fp32` here, because this CPU has no dedicated low-precision matrix
unit to exploit and every operation pays a cast-to-`bfloat16`-and-back cost
with no corresponding speedup underneath it. On a GPU with tensor cores, or a
CPU with AVX-512 BF16/AMX support, the same code would show a speedup instead
— the technique is identical, the result is a property of the hardware it
runs on, which is exactly why it has to be measured rather than assumed.

## Quantising a Model

`torch.quantization.quantize_dynamic` applies the scale/zero-point affine
map 15a derived by hand to a trained model's `nn.Linear` layers, entirely
after training — no retraining, no calibration data required, because
dynamic quantisation computes each layer's activation range on the fly at
inference time and only the *weights* are quantised ahead of time. It is
applied here to a small CNN trained the ordinary way on the CIFAR-10 subset.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # torch.quantization is being migrated to torchao; suppress the (expected) migration notice

quant_model = Net()
opt = torch.optim.Adam(quant_model.parameters(), lr=1e-3)
for epoch in range(15):
    perm = torch.randperm(len(X_train_t))
    for i in range(0, len(X_train_t), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        F.cross_entropy(quant_model(X_train_t[idx]), y_train_t[idx]).backward()
        opt.step()
quant_model.eval()

acc_fp32 = accuracy(quant_model, X_test_t, y_test_t)
qmodel = torch.quantization.quantize_dynamic(quant_model, {nn.Linear}, dtype=torch.qint8)
acc_q = accuracy(qmodel, X_test_t, y_test_t)
print(f"fp32 test accuracy:      {acc_fp32:.3f}")
print(f"quantized test accuracy: {acc_q:.3f}")

In [ ]:
def state_dict_bytes(net):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(net.state_dict(), f.name)
        size = os.path.getsize(f.name)
    os.unlink(f.name)
    return size


def time_inference(net, n=30):
    with torch.no_grad():
        for _ in range(5):
            net(X_test_t)
        t0 = time.perf_counter()
        for _ in range(n):
            net(X_test_t)
        return (time.perf_counter() - t0) / n


size_fp32, size_q = state_dict_bytes(quant_model), state_dict_bytes(qmodel)
lat_fp32, lat_q = time_inference(quant_model), time_inference(qmodel)
print(f"state_dict size: fp32={size_fp32:,} B  quantized={size_q:,} B  reduction={size_fp32/size_q:.2f}x")
print(f"inference latency: fp32={lat_fp32*1000:.2f} ms/batch  quantized={lat_q*1000:.2f} ms/batch  "
      f"ratio={lat_fp32/lat_q:.2f}x ({'faster' if lat_q < lat_fp32 else 'SLOWER'})")

Quantising the model's `Linear` layers costs essentially no accuracy — the
quantized model's test accuracy matches the original within the noise of a
single evaluation — at a real, substantial reduction in on-disk size (the
convolutional layers, left un-quantised here, are why the reduction is well
under the 4x a fully-quantised tensor would show, matching 15a's derivation).
Inference **latency**, however, does *not* improve on this CPU: the
quantize/dequantize bookkeeping dynamic quantisation performs on every call
outweighs the saved arithmetic for a model and batch this small. As with
mixed precision above, the size benefit is unconditional and the speed
benefit is not — real inference speedups from int8 typically show up on
larger models, larger batches, or hardware with dedicated int8 kernels, none
of which apply to this deliberately tiny CPU demonstration.

## Distillation Experiment

A real teacher/student training loop, reproducing 15a's derived loss: a much
larger teacher is trained first, then a small student is trained two ways —
once on hard labels alone, once on the blended distillation loss against the
same teacher — and the two students' test accuracy is compared directly.

In [ ]:
class Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return self.fc(x.flatten(1))


class Student(nn.Module):
    """Deliberately much smaller: one conv layer, fewer channels."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(4)
        self.fc = nn.Linear(8 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        return self.fc(x.flatten(1))


def train_plain(net, epochs, lr=1e-3, batch=64):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X_train_t)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            F.cross_entropy(net(X_train_t[idx]), y_train_t[idx]).backward()
            opt.step()
    return net


def train_distill(net, teacher, epochs, lr=1e-3, batch=64, alpha=0.3, T=4.0):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X_train_t)
    teacher.eval()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            student_logits = net(X_train_t[idx])
            with torch.no_grad():
                teacher_logits = teacher(X_train_t[idx])
            hard = F.cross_entropy(student_logits, y_train_t[idx])
            soft = F.kl_div(F.log_softmax(student_logits / T, 1),
                             F.softmax(teacher_logits / T, 1), reduction="batchmean") * T ** 2
            (alpha * hard + (1 - alpha) * soft).backward()
            opt.step()
    return net


teacher = Teacher()
train_plain(teacher, epochs=15)
n_teacher = sum(p.numel() for p in teacher.parameters())
n_student = sum(p.numel() for p in Student().parameters())
print(f"teacher: {n_teacher:,} params, test accuracy {accuracy(teacher, X_test_t, y_test_t):.3f}")
print(f"student: {n_student:,} params ({n_teacher/n_student:.1f}x fewer than the teacher)")

In [ ]:
ALPHA, T, EPOCHS, N_SEEDS = 0.3, 4.0, 15, 5
acc_plain, acc_distill = [], []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    student_plain = Student()
    train_plain(student_plain, epochs=EPOCHS)
    acc_plain.append(accuracy(student_plain, X_test_t, y_test_t))

    torch.manual_seed(seed)
    student_distill = Student()
    train_distill(student_distill, teacher, epochs=EPOCHS, alpha=ALPHA, T=T)
    acc_distill.append(accuracy(student_distill, X_test_t, y_test_t))

    print(f"seed={seed}  direct-training accuracy={acc_plain[-1]:.3f}  distilled accuracy={acc_distill[-1]:.3f}")

print(f"mean direct-training accuracy: {np.mean(acc_plain):.3f}")
print(f"mean distilled accuracy:       {np.mean(acc_distill):.3f}")

Distilling from the teacher does not produce a clean accuracy win over
training the same small architecture directly on this small, noisy CIFAR-10
subset — the two are close, within the run-to-run noise five different
seeds show. This is reported plainly rather than tuned toward a nicer-looking
story: 15a already verified, directly, what the distillation loss reliably
delivers (a student whose *output distribution* tracks the teacher's much
more closely) — accuracy is one downstream consequence of that, but on a
tiny toy problem it is a noisy one, and a practical notebook's job is to
report the real number, not the expected one.

## Measurements

Every number measured above, collected into one table, so the pattern across
all three techniques is visible at a glance: **the memory/size benefit is
unconditional in every row; the speed benefit is not**.

In [ ]:
summary = pd.DataFrame([
    {"technique": "mixed precision (bf16 autocast)", "metric": "activation memory", "baseline": f"{bytes_fp32:,} B", "technique_value": f"{bytes_bf16:,} B", "ratio": f"{bytes_fp32/bytes_bf16:.2f}x smaller"},
    {"technique": "mixed precision (bf16 autocast)", "metric": "step time", "baseline": f"{t_fp32*1000:.2f} ms", "technique_value": f"{t_bf16*1000:.2f} ms", "ratio": f"{t_fp32/t_bf16:.2f}x ({'faster' if t_bf16 < t_fp32 else 'SLOWER'})"},
    {"technique": "dynamic quantisation", "metric": "state_dict size", "baseline": f"{size_fp32:,} B", "technique_value": f"{size_q:,} B", "ratio": f"{size_fp32/size_q:.2f}x smaller"},
    {"technique": "dynamic quantisation", "metric": "inference latency", "baseline": f"{lat_fp32*1000:.2f} ms", "technique_value": f"{lat_q*1000:.2f} ms", "ratio": f"{lat_fp32/lat_q:.2f}x ({'faster' if lat_q < lat_fp32 else 'SLOWER'})"},
    {"technique": "dynamic quantisation", "metric": "test accuracy", "baseline": f"{acc_fp32:.3f}", "technique_value": f"{acc_q:.3f}", "ratio": f"{acc_q-acc_fp32:+.3f}"},
    {"technique": "distillation", "metric": "student test accuracy", "baseline": f"{np.mean(acc_plain):.3f}", "technique_value": f"{np.mean(acc_distill):.3f}", "ratio": f"{np.mean(acc_distill)-np.mean(acc_plain):+.3f}"},
])
summary

Two rows show an unconditional win (mixed precision's memory, quantisation's
size); two rows show the corresponding speed claim failing on this specific
CPU (mixed precision's step time, quantisation's latency); the accuracy rows
show quantisation costing essentially nothing and distillation landing
within noise. None of this could have been predicted from the techniques'
descriptions alone — it took running each one and measuring the result on
the actual hardware and the actual model.

## Key Takeaways

- **`torch.autocast`'s memory reduction is unconditional (exactly 2x,
  `bfloat16` vs `fp32`) but its speed benefit is not** — measured directly on
  this CPU, autocast ran *slower* than full precision, because this hardware
  has no dedicated low-precision matrix unit to exploit.
- **Dynamic quantisation cost this model essentially no accuracy** at a real
  reduction in on-disk size, but — like mixed precision — **its latency
  benefit did not appear on this CPU**: the quantize/dequantize bookkeeping
  outweighed the saved arithmetic for a model and batch this small.
- **Distillation reproduced with a real training loop did not show a clean
  accuracy win** over training the same small architecture directly, on this
  small, noisy CIFAR-10 subset — reported honestly, because 15a already
  verified what the loss reliably delivers (closer agreement with the
  teacher's output distribution), and accuracy is a separate, noisier
  downstream question.
- **The pattern across all three techniques is the same**: the
  memory/size benefit each technique promises is structural and shows up
  every time; the speed benefit depends on hardware, model size, and batch
  size, and has to be measured on the system that will actually run it — not
  assumed from the technique's name.